In [ ]:
# 1: Install and import dependencies
%pip -q install requests datasets

import json
from pprint import pprint
from typing import List, Dict, Tuple, Any

import requests
from datasets import load_dataset

In [ ]:
# 2: Download the HotpotQA DEV distractor set.

PRIMARY_URL = "http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json"

def load_from_primary(url: str) -> List[Dict[str, Any]]:
    """Try to download and parse the original HotpotQA DEV distractor JSON from the official URL."""
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not isinstance(data, list):
        raise ValueError("Primary URL returned a JSON payload that is not a list of records.")
    return data

def load_from_hf() -> List[Dict[str, Any]]:
    """Load the DEV/validation split of HotpotQA (distractor config) from Hugging Face and convert to a list of dicts."""
    # The 'distractor' config corresponds to the same setting as the official dev distractor file.
    ds = load_dataset("hotpot_qa", "distractor", split="validation")
    # Convert to a plain list of Python dicts to keep a consistent shape with the primary JSON.
    return [dict(x) for x in ds]

def load_hotpotqa_dev_distractor() -> Tuple[List[Dict[str, Any]], str]:
    """
    Attempt to load DEV distractor from the official URL first.
    If that fails, fall back to Hugging Face.
    Returns:
        records: List of example dicts.
        source:  String describing which source was used ("primary" or "huggingface").
    """
    try:
        records = load_from_primary(PRIMARY_URL)
        source = "primary"
    except Exception as e:
        print(f"[Info] Primary download failed ({e}). Falling back to Hugging Face...")
        records = load_from_hf()
        source = "huggingface"
    return records, source

# Perform the load (no files are written; data stays in memory):
records, source_used = load_hotpotqa_dev_distractor()
print(f"Loaded DEV distractor set from: {source_used}")
print(f"Type(records): {type(records).__name__} | Length: {len(records)}")

Loaded DEV distractor set from: primary
Type(records): list | Length: 7405


In [ ]:
# 3: Pretty-print the first three examples (records[0:3]) to familiarize with the structure.

# Defensive checks:
if not isinstance(records, list) or len(records) == 0:
    raise RuntimeError("Dataset appears to be empty or not a list. Check the previous cell output.")

for i, item in enumerate(records[:3], start=1):
    print("=" * 20, f"Sample #{i}", "=" * 20)
    # Use pprint with a wider width so nested fields are more readable.
    pprint(item, width=120, sort_dicts=False)
    print()  # blank line between samples


==================== Sample #1 ====================
{'_id': '5a8b57f25542995d1e6f1371',
 'answer': 'yes',
 'question': 'Were Scott Derrickson and Ed Wood of the same nationality?',
 'supporting_facts': [['Scott Derrickson', 0], ['Ed Wood', 0]],
 'context': [['Ed Wood (film)',
              ['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, '
               'and starring Johnny Depp as cult filmmaker Ed Wood.',
               " The film concerns the period in Wood's life when he made his best-known films as well as his "
               'relationship with actor Bela Lugosi, played by Martin Landau.',
               ' Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the '
               'supporting cast.']],
             ['Scott Derrickson',
              ['Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.',
               ' He lives in Los Angeles, Cal

In [ ]:
# 4: Add a 'titles' column extracted from 'context' without modifying original strings

from typing import Any, Dict, List

def extract_titles(example: Dict[str, Any]) -> List[str]:
    """
    Extract titles from the 'context' field without altering them.
    Supports both shapes:
      - Original JSON: context is List[[title, [sentences...]], ...]
      - HF processed:  context is Dict{'title': List[str], 'sentences': List[List[str]]}
    Returns a list of titles as-is.
    """
    ctx = example.get("context", None)
    if ctx is None:
        return []
    # Shape 1: list of [title, sentences]
    if isinstance(ctx, list):
        out = []
        for elem in ctx:
            if isinstance(elem, (list, tuple)) and len(elem) >= 1:
                out.append(elem[0])
        return out
    # Shape 2: dict with parallel lists
    if isinstance(ctx, dict):
        titles = ctx.get("title", [])
        return list(titles) if isinstance(titles, list) else []
    # Unknown shape
    return []

# Mutate in place: add 'titles' key for each record
for ex in records:
    ex["titles"] = extract_titles(ex)

# Quick sanity print: show how many titles each of the first three items has
print("Added 'titles' to all records. Title counts for first three samples:",
      [len(r.get("titles", [])) for r in records[:3]])


Added 'titles' to all records. Title counts for first three samples: [10, 10, 10]


In [ ]:
# 5: Pretty-print the first three samples focusing on the new 'titles' field

from pprint import pprint

if not isinstance(records, list) or len(records) < 3:
    raise RuntimeError("Not enough records to display. Ensure previous cells ran successfully.")

for i, item in enumerate(records[:3], start=1):
    print("=" * 20, f"Sample #{i}", "=" * 20)
    pprint({"_id": item.get("_id"), "titles": item.get("titles")}, width=120, sort_dicts=False)
    print()


==================== Sample #1 ====================
{'_id': '5a8b57f25542995d1e6f1371',
 'titles': ['Ed Wood (film)',
            'Scott Derrickson',
            'Woodson, Arkansas',
            'Tyler Bates',
            'Ed Wood',
            'Deliver Us from Evil (2014 film)',
            'Adam Collis',
            'Sinister (film)',
            'Conrad Brooks',
            'Doctor Strange (2016 film)']}

==================== Sample #2 ====================
{'_id': '5a8c7595554299585d9e36b6',
 'titles': ['Meet Corliss Archer',
            'Shirley Temple',
            'Janet Waldo',
            'Meet Corliss Archer (TV series)',
            'Lord High Treasurer',
            'A Kiss for Corliss',
            'Kiss and Tell (1945 film)',
            'Secretary of State for Constitutional Affairs',
            'Village accountant',
            'Charles Craft']}

==================== Sample #3 ====================
{'_id': '5a85ea095542994775f606a8',
 'titles': ['Andre Norton Award',
   

In [ ]:
# 6: Load the official 2017-10-01 Wikipedia snapshot used by HotpotQA (local dump, not live API)

%pip -q install tqdm

import os
import tarfile
import bz2
import html
import unicodedata
import re
from pathlib import Path
from typing import Optional

from tqdm.auto import tqdm

# According to the official HotpotQA documentation, the dataset is built from the
# English Wikipedia dump dated 2017-10-01, preprocessed and hosted by the authors:
#   https://hotpotqa.github.io/wiki-readme.html
#
# The preprocessed Wikipedia for HotpotQA is distributed at:
#   https://nlp.stanford.edu/projects/hotpotqa/enwiki-20171001-pages-meta-current-withlinks-processed.tar.bz2
#
# We download this archive once and work entirely locally, thereby avoiding
# (i) API rate limiting ("429 Too Many Requests") and
# (ii) temporal mismatch between 2017 content and the current live Wikipedia.

WIKI_ARCHIVE_URL = (
    "https://nlp.stanford.edu/projects/hotpotqa/"
    "enwiki-20171001-pages-meta-current-withlinks-processed.tar.bz2"
)
WIKI_ARCHIVE_PATH = Path("enwiki-20171001-pages-meta-current-withlinks-processed.tar.bz2")
WIKI_ROOT_DIR = Path("enwiki-20171001-pages-meta-current-withlinks-processed")


def download_wiki_archive(url: str, dest: Path) -> None:
    """
    Download the HotpotQA preprocessed Wikipedia archive if it is not present locally.

    The file is ~7.4 GB compressed, so this step is intended to be run once and
    the resulting artifact cached on disk (e.g., in a Colab or cluster filesystem).
    """
    if dest.is_file():
        print(f"[Info] {dest.name} already present, skipping download.")
        return

    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"[Info] Downloading preprocessed Wikipedia from: {url}")
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(
            total=total, unit="B", unit_scale=True, desc=f"Downloading {dest.name}"
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if not chunk:
                    continue
                f.write(chunk)
                pbar.update(len(chunk))


def extract_wiki_archive(archive_path: Path, root_dir: Path) -> None:
    """
    Extract the top-level tar.bz2 archive into a directory of .bz2 shards.

    The resulting directory structure matches the one described in the HotpotQA
    wiki-readme: numerous subdirectories each containing a few .bz2 files, where
    each .bz2 holds multiple JSON objects (one per Wikipedia page).
    """
    if root_dir.is_dir():
        print(f"[Info] {root_dir} already extracted, skipping extraction.")
        return

    print(f"[Info] Extracting {archive_path.name} (this may take several minutes)...")
    with tarfile.open(archive_path, mode="r:bz2") as tar:
        members = tar.getmembers()
        for m in tqdm(members, desc="Extracting Wikipedia shards"):
            tar.extract(m, path=".")


def normalize_title_key(title: str) -> str:
    """
    Normalization function for matching titles between HotpotQA and the wiki dump.

    Operations:
      - Decode HTML entities (e.g. '&amp;' → '&').
      - Normalize Unicode to NFC.
      - Replace underscores with spaces.
      - Trim leading/trailing whitespace.
      - Case-fold (a stronger form of lowercasing).

    This mirrors the fact that Wikipedia titles are case-insensitive and that
    HotpotQA sometimes encodes special characters (e.g. '&amp;' in titles).
    """
    if not isinstance(title, str):
        title = str(title)
    title = html.unescape(title)
    title = unicodedata.normalize("NFC", title)
    title = title.replace("_", " ").strip()
    return title.casefold()


def normalize_text(text: str) -> str:
    """
    Normalize free text for robust substring matching between HotpotQA contexts
    and Wikipedia dump articles.

    Transformations:
      - HTML entity decoding (e.g. '&amp;' → '&').
      - Unicode NFKC normalization.
      - Canonicalization of spaces around punctuation:
          * Remove spaces directly before , . ; : ! ?
          * Remove inner spaces right after '(' or '[' and right before ')' or ']'
      - Lowercasing.
      - Collapsing runs of whitespace (spaces, newlines, tabs) into a single space.
      - Trimming leading/trailing whitespace.

    By applying the same normalization to both contexts and Wikipedia texts,
    we make matching robust to idiosyncrasies introduced by hyperlink markup
    and sentence segmentation in the processed dump.
    """
    if not isinstance(text, str):
        text = str(text)

    # Decode HTML entities first (important for '&amp;' etc.).
    text = html.unescape(text)

    # Unicode canonicalization.
    text = unicodedata.normalize("NFKC", text)

    # Remove spaces immediately before common punctuation.
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)

    # Clean up spaces inside parentheses and brackets.
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    text = re.sub(r"\[\s+", "[", text)
    text = re.sub(r"\s+\]", "]", text)

    # Lowercase and collapse whitespace.
    text = text.lower()
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# Trigger the download + extraction steps once.
download_wiki_archive(WIKI_ARCHIVE_URL, WIKI_ARCHIVE_PATH)
extract_wiki_archive(WIKI_ARCHIVE_PATH, WIKI_ROOT_DIR)


[Info] Downloading preprocessed Wikipedia from: https://nlp.stanford.edu/projects/hotpotqa/enwiki-20171001-pages-meta-current-withlinks-processed.tar.bz2


[Info] Extracting enwiki-20171001-pages-meta-current-withlinks-processed.tar.bz2 (this may take several minutes)...


Extracting Wikipedia shards:   0%|          | 0/15674 [00:00<?, ?it/s]

/tmp/ipython-input-3771967140.py:77: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(m, path=".")


In [7]:
# 7: Build a mapping from all HotpotQA DEV titles to their 2017 Wikipedia articles

import json
import bz2
from typing import Dict, Any, List, Set

# Collect all distinct titles that appear anywhere in the DEV split.
all_titles: List[str] = sorted({t for ex in records for t in ex.get("titles", [])})
print(f"Total unique HotpotQA titles in DEV split: {len(all_titles)}")

# Map normalized title → one canonical HotpotQA title string.
norm_to_original: Dict[str, str] = {}
for t in all_titles:
    key = normalize_title_key(t)
    # If collisions happen (unlikely), we deterministically keep the first mapping.
    norm_to_original.setdefault(key, t)

remaining: Set[str] = set(norm_to_original.keys())

# This dictionary will hold the full JSON article objects from the 2017 dump.
WIKI_ARTICLES: Dict[str, Dict[str, Any]] = {}

bz2_files = sorted(WIKI_ROOT_DIR.rglob("*.bz2"))
print(f"Found {len(bz2_files)} compressed shards in the 2017 Wikipedia dump.")

# Single pass over the dump: whenever we see a title we care about, store the article JSON.
for shard_path in tqdm(bz2_files, desc="Scanning Wikipedia shards for all DEV titles"):
    if not remaining:
        break  # All desired titles have been resolved.

    with bz2.open(shard_path, mode="rt", encoding="utf-8") as f:
        for line in f:
            if not remaining:
                break
            obj = json.loads(line)
            title_raw = obj.get("title", "")
            norm = normalize_title_key(title_raw)
            if norm in remaining:
                original_title = norm_to_original[norm]
                WIKI_ARTICLES[original_title] = obj
                remaining.remove(norm)

print(f"\nRecovered {len(WIKI_ARTICLES)} / {len(all_titles)} titles from the 2017 dump.")

if remaining:
    missing_titles = [norm_to_original[n] for n in sorted(remaining)]
    print("Example titles not found in the local 2017 dump (showing up to 20):")
    for t in missing_titles[:20]:
        print("  -", t)


Total unique HotpotQA titles in DEV split: 66581
Found 15517 compressed shards in the 2017 Wikipedia dump.


Scanning Wikipedia shards for all DEV titles:   0%|          | 0/15517 [00:00<?, ?it/s]


Recovered 66576 / 66581 titles from the 2017 dump.


In [8]:
# 8: Build cleaned plaintext representations of all recovered Wikipedia pages
#     - Skip the title "paragraph" from the dump.
#     - Remove links/tags more carefully to avoid extra spaces.
#     - ALSO build a sentence-level variant per article (for 'docs2' later).

from typing import Dict, List, Tuple, Any

def is_title_paragraph(para, title: str) -> bool:
    """
    Heuristic: check if the first paragraph in article["text"] is just the page title.

    Many pages in the HotpotQA wiki dump have the title as a one-sentence 'paragraph'
    at index 0, e.g. ["Dime Magazine"]. We don't want to duplicate this in docs,
    because the title is already available separately.

    We:
      - take the first sentence of `para`
      - strip inline HTML tags
      - unescape HTML entities
      - normalize spaces and case
      - compare it with a normalized version of the article title
    """
    if not isinstance(para, list) or not para:
        return False

    first_sent = para[0]
    if not isinstance(first_sent, str):
        return False

    s = first_sent.strip()
    if not s:
        return False

    # Remove only the tags, keep the inner text.
    s = re.sub(r"<a[^>]*>", "", s)    # opening <a ...>
    s = re.sub(r"</a>", "", s)        # closing </a>
    # Remove any other tags (e.g., <sup>, <span>, etc.) but keep spacing.
    s = re.sub(r"<[^>]+>", " ", s)

    # Decode HTML entities and normalize spaces.
    s = html.unescape(s)
    s = re.sub(r"\s+", " ", s).strip()

    # Normalize title in a way consistent with our title matching logic.
    title_norm = normalize_title_key(title)          # already casefolded, trimmed, spaces normalized
    s_norm = s.casefold()                            # lower/casefold for comparison

    return s_norm == title_norm


def clean_sentence(s: str) -> str:
    """
    Clean a single sentence string from the wiki dump:

      - strip leading/trailing whitespace
      - remove <a ...>...</a> tags but keep the inner text
      - remove other HTML tags like <sup>...</sup> (keeping spacing)
      - decode HTML entities
      - collapse multiple spaces into one
    """
    if not isinstance(s, str):
        return ""

    s = s.strip()
    if not s:
        return ""

    # Remove link tags but keep anchor text.
    s = re.sub(r"<a[^>]*>", "", s)    # remove opening <a ...>
    s = re.sub(r"</a>", "", s)        # remove closing </a>

    # Remove other tags (e.g., <sup>, <span>, etc.) and replace with a single space
    # so text on both sides doesn't stick together.
    s = re.sub(r"<[^>]+>", " ", s)

    # Decode HTML entities (e.g. &amp; → &).
    s = html.unescape(s)

    # Normalize internal whitespace.
    s = re.sub(r"\s+", " ", s).strip()

    return s


def article_json_to_plaintext_and_sentences(article: Dict[str, Any]) -> Tuple[str, List[str]]:
    """
    Convert a single Wikipedia JSON object from the HotpotQA dump into:

      - a linear plaintext string suitable for attaching as a full document (`docs`)
      - a flat list of cleaned sentences in article order (for `docs2`).

    The relevant field is:
        article["text"] : List[List[str]]
            - Outer list: paragraphs
            - Inner list: sentences in that paragraph

    Steps:
      - Optionally skip the first "paragraph" if it is just the page title.
      - For each remaining paragraph:
          * clean each sentence with `clean_sentence`
          * append sentence to a global sentence list
          * join sentences with a single space to form the paragraph text
      - Join paragraphs with a blank line separator ("\n\n") for the full text.
    """
    paragraphs = article.get("text", [])
    title = article.get("title", "")

    if not isinstance(paragraphs, list):
        paragraphs = []

    pieces: List[str] = []         # paragraph strings for the full doc
    all_sentences: List[str] = []  # flat list of cleaned sentences

    # Decide whether to skip the first paragraph (if it looks like the title only).
    start_idx = 0
    if paragraphs and is_title_paragraph(paragraphs[0], title):
        start_idx = 1

    for para in paragraphs[start_idx:]:
        if not isinstance(para, list):
            continue

        # Clean each sentence and keep non-empty results
        sent_clean: List[str] = []
        for s in para:
            cs = clean_sentence(s)
            if cs:
                sent_clean.append(cs)
                all_sentences.append(cs)  # keep exact sentence text for docs2

        if not sent_clean:
            continue

        # Join sentences with exactly one space.
        para_str = " ".join(sent_clean)

        # As a last safety, collapse any accidental multiple spaces again.
        para_str = re.sub(r"\s+", " ", para_str).strip()

        if para_str:
            pieces.append(para_str)

    # Separate paragraphs by blank lines for readability in 'docs'.
    full = "\n\n".join(pieces)

    return full, all_sentences


# Full article text and sentence-level lists for every recovered title.
WIKI_ARTICLE_TEXT: Dict[str, str] = {}          # title -> full paragraph-style text (for 'docs')
WIKI_ARTICLE_SENTENCES: Dict[str, List[str]] = {}  # title -> list of cleaned sentences (for 'docs2')

for title, article in WIKI_ARTICLES.items():
    full_text, sentence_list = article_json_to_plaintext_and_sentences(article)
    WIKI_ARTICLE_TEXT[title] = full_text
    WIKI_ARTICLE_SENTENCES[title] = sentence_list

print(
    f"Constructed cleaned article texts for {len(WIKI_ARTICLE_TEXT)} / {len(all_titles)} titles, "
    f"with sentence lists for {len(WIKI_ARTICLE_SENTENCES)} titles."
)

Constructed cleaned article texts for 66576 / 66581 titles, with sentence lists for 66576 titles.


In [9]:
# 9: Select 700 bridge + 300 comparison examples whose titles have
#    Wikipedia docs AND whose context paragraphs match those docs.

from dataclasses import dataclass
from typing import List as _ListType, Dict as _DictType, Any
import re
import html
import unicodedata


def normalize_for_match(text: str) -> str:
    """Normalize text for matching: ignore case, punctuation, HTML tags and extra spaces."""
    if not isinstance(text, str):
        text = str(text)
    # Decode HTML entities
    text = html.unescape(text)
    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)
    # Remove HTML-like tags (e.g. <nowiki>, <ref>, ...)
    text = re.sub(r"<[^>]+>", " ", text)
    # Lowercase
    text = text.lower()
    # Keep only letters and digits, turn others into spaces
    text = re.sub(r"[^0-9a-z0-9]+", " ", text)
    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text


def example_has_all_consistent_docs(example: Dict[str, Any]) -> bool:
    """
    For this example:
      - every title must have a wiki doc in WIKI_ARTICLE_TEXT
      - the context paragraph for each title must be contained
        in the doc (after normalization).
    """
    titles = example.get("titles", [])
    context = example.get("context", [])

    if not titles or not context:
        return False

    # Build title -> context paragraph map
    context_map: Dict[str, str] = {}
    for item in context:
        # Each item should be [title, [sent_0, sent_1, ...]]
        if isinstance(item, (list, tuple)) and len(item) == 2:
            ctx_title, sent_list = item
            if isinstance(ctx_title, str) and isinstance(sent_list, list):
                paragraph = " ".join(
                    s.strip() for s in sent_list
                    if isinstance(s, str) and s.strip()
                )
                if paragraph:
                    # Keep the first paragraph seen for this title
                    context_map.setdefault(ctx_title, paragraph)

    # Check consistency for each title
    for t in titles:
        doc_text = WIKI_ARTICLE_TEXT.get(t)
        if not doc_text:
            # No wiki doc for this title
            return False

        ctx_para = context_map.get(t)
        if not ctx_para:
            # No context paragraph for this title
            return False

        norm_ctx = normalize_for_match(ctx_para)
        norm_doc = normalize_for_match(doc_text)

        # If normalized context is not a substring of normalized doc → inconsistent
        if norm_ctx and norm_ctx not in norm_doc:
            return False

    return True


@dataclass
class SelectionStats:
    """Summary statistics for the doc-availability + consistency-based selection."""
    total_bridge: int
    total_comparison: int
    eligible_bridge: int
    eligible_comparison: int
    selected_bridge: int
    selected_comparison: int


# Partition DEV questions by type and require at least 10 titles.
bridge_candidates: List[Dict[str, Any]] = [
    ex for ex in records
    if ex.get("type") == "bridge" and len(ex.get("titles", [])) >= 10
]
comparison_candidates: List[Dict[str, Any]] = [
    ex for ex in records
    if ex.get("type") == "comparison" and len(ex.get("titles", [])) >= 10
]

print(f"Total bridge questions in DEV with ≥10 titles:      {len(bridge_candidates)}")
print(f"Total comparison questions in DEV with ≥10 titles: {len(comparison_candidates)}")

# Keep only questions whose titles all have consistent docs
eligible_bridge: List[Dict[str, Any]] = [
    ex for ex in bridge_candidates if example_has_all_consistent_docs(ex)
]
eligible_comparison: List[Dict[str, Any]] = [
    ex for ex in comparison_candidates if example_has_all_consistent_docs(ex)
]

print(f"Eligible bridge questions (all titles have consistent docs):      {len(eligible_bridge)}")
print(f"Eligible comparison questions (all titles have consistent docs): {len(eligible_comparison)}")

target_bridge = 700
target_comparison = 300

# Select up to the target numbers (if there are fewer eligible, we take as many as possible).
selected_bridge: List[Dict[str, Any]] = eligible_bridge[:target_bridge]
selected_comparison: List[Dict[str, Any]] = eligible_comparison[:target_comparison]

stats = SelectionStats(
    total_bridge=len(bridge_candidates),
    total_comparison=len(comparison_candidates),
    eligible_bridge=len(eligible_bridge),
    eligible_comparison=len(eligible_comparison),
    selected_bridge=len(selected_bridge),
    selected_comparison=len(selected_comparison),
)

print("\nSelection summary:")
print(f"  Eligible bridge questions:      {stats.eligible_bridge}")
print(f"  Eligible comparison questions: {stats.eligible_comparison}")
print(f"  Selected bridge questions (target {target_bridge}):      {stats.selected_bridge}")
print(f"  Selected comparison questions (target {target_comparison}): {stats.selected_comparison}")

# Concatenate the final subset (bridge first, then comparison).
selected_all: List[Dict[str, Any]] = selected_bridge + selected_comparison
print(f"\nTotal selected questions: {len(selected_all)}")

# Quick sanity check of the distribution.
num_bridge_in_selected = sum(1 for ex in selected_all if ex.get("type") == "bridge")
num_comparison_in_selected = sum(1 for ex in selected_all if ex.get("type") == "comparison")
print(f"  Breakdown in selected_all: {num_bridge_in_selected} bridge, {num_comparison_in_selected} comparison")

Total bridge questions in DEV with ≥10 titles:      5899
Total comparison questions in DEV with ≥10 titles: 1446
Eligible bridge questions (all titles have consistent docs):      5855
Eligible comparison questions (all titles have consistent docs): 1418

Selection summary:
  Eligible bridge questions:      5855
  Eligible comparison questions: 1418
  Selected bridge questions (target 700):      700
  Selected comparison questions (target 300): 300

Total selected questions: 1000
  Breakdown in selected_all: 700 bridge, 300 comparison


In [10]:
# 10: Attach full Wikipedia documents as 'docs' + sentence-level 'docs2'
#     and write the 1000-question subset to JSON.

from pathlib import Path
from typing import List, Dict, Any

OUTPUT_JSON_PATH = Path("hotpotqa_dev_2017wiki_1000.json")

def attach_docs_field(example: Dict[str, Any]) -> Dict[str, Any]:
    """
    Create a shallow copy of `example` and attach two fields:

      - 'docs'  : list[str]
          Full article text (paragraph-style), one string per title.
      - 'docs2' : list[list[str]]
          Sentence-level text; for each title, a list of cleaned sentences
          in article order.

    Both are aligned with the HotpotQA 'titles' field:

        example['titles'][i]  <->  docs[i], docs2[i]

    Precondition: every title in example['titles'] has corresponding entries
    in WIKI_ARTICLE_TEXT and WIKI_ARTICLE_SENTENCES (enforced earlier by
    the consistency checks when selecting `selected_all`).
    """
    titles = example.get("titles", [])
    docs: List[str] = []
    docs2: List[List[str]] = []

    for t in titles:
        doc_text = WIKI_ARTICLE_TEXT.get(t)
        sent_list = WIKI_ARTICLE_SENTENCES.get(t)

        if doc_text is None or sent_list is None:
            # Defensive check: should not happen if example_has_all_consistent_docs passed.
            raise KeyError(
                f"Missing Wikipedia data for title {t!r} when building 'docs'/'docs2' fields."
            )

        docs.append(doc_text)
        # Store a shallow copy of the sentence list to avoid accidental mutation downstream.
        docs2.append(list(sent_list))

    ex_out = dict(example)  # shallow copy to avoid mutating the original record list
    ex_out["docs"] = docs
    ex_out["docs2"] = docs2
    return ex_out


# Build the final subset with attached docs + docs2.
selected_with_docs: List[Dict[str, Any]] = [attach_docs_field(ex) for ex in selected_all]

# Sanity check: count how many bridge/comparison examples ended up in the final JSON.
num_bridge_out = sum(1 for ex in selected_with_docs if ex.get("type") == "bridge")
num_comparison_out = sum(1 for ex in selected_with_docs if ex.get("type") == "comparison")
print(f"Final subset sizes in JSON: {num_bridge_out} bridge, {num_comparison_out} comparison.")
print(f"Total written examples: {len(selected_with_docs)}")

# Write to disk as a pretty-printed UTF-8 JSON file.
with OUTPUT_JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(selected_with_docs, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(selected_with_docs)} questions with attached Wikipedia docs (docs + docs2) to {OUTPUT_JSON_PATH}")

Final subset sizes in JSON: 700 bridge, 300 comparison.
Total written examples: 1000
Wrote 1000 questions with attached Wikipedia docs (docs + docs2) to hotpotqa_dev_2017wiki_1000.json


In [14]:
# 11: Mount Google Drive and copy the JSON file into it

from google.colab import drive
import os
import shutil

# Mount Google Drive at /content/drive
drive.mount('/content/drive')

# Local path of the JSON file created earlier
local_json_path = "hotpotqa_dev_2017wiki_1000.json"

# Target folder inside your Google Drive (you can change this path if you like)
drive_folder = "/content/drive/MyDrive/hotpotqa_data"

# Make sure the target folder exists
os.makedirs(drive_folder, exist_ok=True)

# Full destination path in Google Drive
drive_json_path = os.path.join(drive_folder, os.path.basename(local_json_path))

# Copy the file from the local environment to Google Drive
shutil.copy2(local_json_path, drive_json_path)

print(f"Copied {local_json_path} → {drive_json_path}")


Mounted at /content/drive
Copied hotpotqa_dev_2017wiki_1000.json → /content/drive/MyDrive/hotpotqa_data/hotpotqa_dev_2017wiki_1000.json


In [15]:
# Check all selected_with_docs: for each title, see if normalized context is contained in normalized doc.
# If not, print the mismatches.

from pprint import pprint
import html
import unicodedata
import re

def normalize_for_match_local(text: str) -> str:
    """Normalize text for matching: ignore case, punctuation, HTML tags and extra spaces."""
    if not isinstance(text, str):
        text = str(text)
    # Decode HTML entities
    text = html.unescape(text)
    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)
    # Remove HTML-like tags (e.g. <nowiki>, <ref>, ...)
    text = re.sub(r"<[^>]+>", " ", text)
    # Lowercase
    text = text.lower()
    # Keep only letters and digits, turn others into spaces
    text = re.sub(r"[^0-9a-z0-9]+", " ", text)
    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Use the 1000-example subset built in cell 10
examples = selected_with_docs

mismatches = []

for ex_idx, ex in enumerate(examples):
    # Build context title -> paragraph map for this example
    context_map = {}
    for item in ex.get("context", []):
        # Each item should be [title, [sent_0, sent_1, ...]]
        if isinstance(item, (list, tuple)) and len(item) == 2:
            ctx_title, sent_list = item
            if isinstance(ctx_title, str) and isinstance(sent_list, list):
                paragraph = " ".join(
                    s.strip() for s in sent_list
                    if isinstance(s, str) and s.strip()
                )
                if paragraph:
                    context_map.setdefault(ctx_title, paragraph)

    titles = ex.get("titles", [])
    docs = ex.get("docs", [])

    for i, (t, d) in enumerate(zip(titles, docs)):
        ctx_para = context_map.get(t)
        if not ctx_para:
            # No context paragraph for this title in this example
            continue

        norm_ctx = normalize_for_match_local(ctx_para)
        norm_doc = normalize_for_match_local(d)

        if norm_ctx and norm_ctx not in norm_doc:
            mismatches.append({
                "example_index": ex_idx,
                "_id": ex.get("_id"),
                "title": t,
                "question": ex.get("question"),
                "context_raw": ctx_para,
                "doc_raw": d,
            })

print(f"Total title-level mismatches (normalized context not found in doc): {len(mismatches)}")

# Print details for all mismatches (if any)
for m in mismatches:
    print("=" * 80)
    print(f"Example index: {m['example_index']}  |  _id: {m['_id']}")
    print(f"Title: {m['title']}")
    print(f"Question: {m['question']}")
    print("- Context paragraph:")
    print(m["context_raw"])
    print("- Doc text (snippet):")
    doc_snip = m["doc_raw"].replace("\n", " ")
    if len(doc_snip) > 800:
        doc_snip = doc_snip[:800] + "..."
    print(doc_snip)
    print("=" * 80)


Total title-level mismatches (normalized context not found in doc): 0
